# Lab 2: a reward model for a language model

**RLHF course, session 2 · Pegah Alizadeh**

This lab follows Chapter 5 of Nathan Lambert's [RLHF book](https://rlhfbook.com), and its
[suggested experiment](https://rlhfbook.com/c/05-reward-models#suggested-experiments): train a Bradley-Terry reward model on UltraFeedback. It is adapted
from the book's [`code/reward_models`](https://github.com/natolambert/rlhf-book/tree/main/code/reward_models) example.

**Goal:** turn a small language model into a **reward model**, a model that reads a prompt and a response and outputs
one number, trained so that preferred responses get higher numbers. Then look at what it learned, with statistics.

| Part | What you do | Time |
|---|---|---|
| 1 | Look at real preference data | 10 min |
| 2 | Build the reward model: read one score per response (exercise A) and write the loss (exercise B) | 20 min |
| 3 | Train on 3,000 pairs | 15 min (it runs while you discuss) |
| 4 | Evaluate: accuracy, a length baseline with a paired comparison (exercise C), rating gaps, calibration and length bias | 20 min |
| 5 | Score your own responses, and try to fool the model | 10 min |

If you get stuck, each exercise has a hint and a solution you can open. The aim is to understand the steps, so use them.

**After class:** [HW1](https://github.com/pegahani/rlhf-course) studies the same Bradley-Terry model from scratch, where the
truth is known: maximum likelihood, Fisher information and confidence intervals. It is not graded.

**Sources.** Lambert, Chapter 5: [Training a Bradley-Terry reward model](https://rlhfbook.com/c/05-reward-models#training-a-bradley-terry-reward-model),
[Architecture](https://rlhfbook.com/c/05-reward-models#the-default-reward-model-architecture) and [Implementation example](https://rlhfbook.com/c/05-reward-models#implementation-example).
Lambert, Chapter 11, [Bias in data collection](https://rlhfbook.com/c/11-preference-data#bias-things-to-watch-out-for-in-data-collection). Truong, Haupt and
Koyejo, *Machine Learning from Human Preferences*, [Section 2.9](https://mlhp.stanford.edu/src/chap2.html#sec-real-world) (LLM preference learning,
uniform and systematic noise). The statistical analyses in Part 4 were written for this course.

## The big picture

InstructGPT's recipe has three steps. Lab 1 was step 1, instruction tuning. **Today is step 2:**

1. **Instruction tuning:** teach a base model to answer
2. **Reward model:** learn a score $r(x, y)$ for a prompt $x$ and a response $y$, from comparisons. **This is today's lab.**
3. **Reinforcement learning:** train the model to write responses with a high score.

The model behind step 2 is **Bradley-Terry**, from the lecture:

$$
P(y_w \succ y_l \mid x) = \sigma\left(r(x, y_w) - r(x, y_l)\right),
$$

where $y_w$ is the preferred response (the winner, or **chosen**) and $y_l$ the other one (the loser, or **rejected**).
In the lecture, the strengths were free parameters. Here the strength is computed by a **neural network** from the text,
so the model can score responses it has never seen.

### Key words

| Word | Meaning |
|---|---|
| **Preference pair** | A prompt with two responses, one marked chosen and one rejected |
| **Reward model** | A language model with a small extra layer, the **head**, that outputs one number instead of next-token scores |
| **Hidden state** | The vector the model computes at each position, before any output layer. Here, 1,024 numbers per token. |
| **Reward margin** | $r(x, y_w) - r(x, y_l)$: positive when the model agrees with the label |
| **Accuracy** | The fraction of pairs where the margin is positive |
| **Calibration** | Whether a model that says "80% sure" is right about 80% of the time |

## Before you start

- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2**. Turn on **Internet** in the same panel.
- If you have no GPU, ask to pair with a neighbor.
- **Out of memory?** Restart the session (Colab: Runtime → Restart session; Kaggle: Run → Restart & clear cell outputs), then run all the cells again from the top.

**This cell:** installs `transformers` (the language models and their tokenizers) and `datasets` (to download the preference data), only if they are missing or too old.

**You should see:** nothing, or a few lines from pip. Red "dependency resolver" warnings are harmless.

In [ ]:
# Colab and Kaggle already include PyTorch. This installs the other libraries only if they are missing or too old,
# so it usually takes a few seconds. (transformers 4.56 or later is needed for the dtype= argument used below.)
%pip install -q "transformers>=4.56" datasets

**This cell:** loads the libraries: `torch` (tensors and training), `transformers` (the model and its tokenizer), `datasets` (the preference data) and `matplotlib` (plots). Then it checks whether a GPU is available.

**You should see:** `Device: Tesla T4`. If you see `CPU`, change the runtime type (see "Before you start").

In [ ]:
import gc
import math
import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (training will be very slow)")

**This cell:** puts every setting in one place, and fixes the random seed so that everyone gets similar results. The table below explains each setting.

**You should see:** no output.

In [ ]:
# Everything you may want to change later is here.
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"     # the book's choice: a 0.6B base model
DATASET_NAME = "argilla/ultrafeedback-binarized-preferences-cleaned"

N_TRAIN = 3000       # the book's run uses 5,000 pairs
N_TEST = 500         # held-out pairs, never used for training
MAX_LENGTH = 512     # pairs with a longer conversation are dropped (the book uses 512 too)
BATCH_SIZE = 4       # pairs per forward pass, so 8 sequences
ACCUM_STEPS = 4      # effective batch size = 4 x 4 = 16 pairs
EPOCHS = 1           # reward models are usually trained for a single epoch
LR = 2e-5
TRAIN_LAST_LAYERS = 8   # of 28: the rest stay frozen, so that training is fast on a free T4
MAX_MINUTES = 15     # training stops cleanly at this limit
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

**What these settings mean.** You don't need to change them today. They return in "Going further" at the end.

| Setting | Meaning |
|---|---|
| `N_TRAIN`, `N_TEST` | Number of preference pairs to train on, and to test on |
| `MAX_LENGTH` | Longest conversation we keep, in tokens |
| `BATCH_SIZE`, `ACCUM_STEPS` | 4 pairs per forward pass, 4 passes per update: **16 pairs per update** |
| `EPOCHS` | How many times we go through the training pairs |
| `LR` | Learning rate: how big each update is |
| `TRAIN_LAST_LAYERS` | How many of the model's 28 layers we train. The others stay frozen. |
| `MAX_MINUTES` | Training stops after this time, even if it hasn't finished |
| `SEED` | Fixes the randomness, so that everyone gets similar results |

## Part 1: real preference data

**UltraFeedback** collected about 64,000 prompts. For each prompt, several language models wrote a response, and
**GPT-4 rated each response** from 1 to 5 on instruction following, truthfulness, honesty and helpfulness. This version,
cleaned by Argilla, keeps one pair per prompt: the best-rated response as **chosen** and a lower rated one as
**rejected**, with the average ratings.

So these labels are **AI feedback**, not human feedback. That's common today, as we saw in the lecture, and it brings
its own biases.

**This cell:** downloads UltraFeedback (about a minute the first time), then prints one pair: the prompt, the chosen response with its rating and the model that wrote it, and the same for the rejected response. Long texts are cut after 300 or 400 characters, to keep the output short.

**You should see:** about 61,000 rows, and two responses with quite different ratings.

In [ ]:
dataset = load_dataset(DATASET_NAME, split="train")
print(dataset)

example = dataset[0]
print("\nPROMPT:", example["prompt"][:300])
print(f"\nCHOSEN   (rating {example['chosen-rating']}, written by {example['chosen-model']}):")
print(example["chosen"][-1]["content"][:400])
print(f"\nREJECTED (rating {example['rejected-rating']}, written by {example['rejected-model']}):")
print(example["rejected"][-1]["content"][:400])

**Discuss with your neighbor:** do you agree with the label? Is the chosen response better, or just longer, or more
confident? Look at one or two more pairs by changing the index.

<details><summary>Possible answers</summary>

There is no single right answer, which is the point. Things people usually notice:

- The chosen response is often **longer and more detailed**. Sometimes that's genuinely better, sometimes it's padding.
- Ratings are an **average over four aspects**, so a response can win on helpfulness while being less honest.
- Some labels look **wrong or arbitrary**. GPT-4 is a noisy annotator, like a person.

Keep your examples in mind: Part 4 measures how much of the model's accuracy comes from length alone.
</details>

### From a pair to tokens

Each response is stored as a conversation, a list of messages, just as in Lab 1. The reward model reads the **whole
conversation**, prompt and response, through the chat template. The cell below prints the beginning of one.

**This cell:** downloads Qwen3's tokenizer and prints the beginning of one conversation exactly as the reward model will read it. The tokenizer has no padding token, so we reuse its end-of-text token for padding.

**You should see:** the chat-template markers, explained right after the output.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.apply_chat_template(example["chosen"], tokenize=False)[:600])

**What you see.** The dataset only stores the messages. Everything else is added by the **chat template**:

- `<|im_start|>user` and `<|im_start|>assistant`: a message starts, and the word after it says who speaks.
- `<|im_end|>`: the message ends.
- `<think>` ... `</think>`: Qwen3's slot for reasoning before the answer. It's empty here, because these responses
  contain no reasoning.

A few things to know:

- **Each marker is one special token,** with its own id, not a string of characters. Our model is a **base** model,
  not instruction-tuned, so for it these markers are simply a clear, consistent way to separate the prompt from the
  response. The reward model learns to use them during training.
- **Every model family has its own template.** OLMo 2 in Lab 1 used `<|user|>` and `<|assistant|>`, with no `<think>`
  block, because it was trained to answer directly. Qwen3 can either reason first or answer directly, so its template
  always keeps a place for reasoning, even when it's empty.
- **It doesn't bias the comparison:** the chosen and the rejected responses get exactly the same markers and the same
  empty block, so only the answers differ.
- **Never write these markers by hand.** Always use `tokenizer.apply_chat_template`: it uses the format the model was
  trained on, and changing the model only means changing `MODEL_NAME`.

The output stops after 600 characters (`[:600]`), so you may not see the answer. In full, the conversation ends with the
answer, `<|im_end|>` and a newline. That last token is where the reward model will read its score.

`encode_pair` below turns one row into two lists of token ids, one for the chosen conversation and one for the rejected
one. It also keeps the rating gap and the length of each response, in tokens, for Part 4.

Pairs where either conversation is longer than `MAX_LENGTH` are **dropped, not cut**: the reward is read at the last token,
and cutting would hide the end of the response. About 40% of the pairs are dropped, the longest ones, so our model
only ever sees shorter responses. Keep this in mind when you read the results.

**This cell:** prepares all the data, in three steps.

1. **Three functions.** `encode` turns a conversation into token ids, through the chat template. `response_length` counts the tokens of the response alone (for Part 4). `encode_pair` turns one dataset row into both conversations as token ids, plus the rating gap and both lengths; it returns `None` for a pair we drop.
2. **Encode.** Shuffle the dataset and encode $3 \times 3{,}500 = 10{,}500$ rows: three times what we need, because about 40% will be dropped as too long.
3. **Split.** The first 500 surviving pairs become the **test set**, the next 3,000 the **training set**.

**You should see:** `Kept about 6,000 of 10,500 pairs ... 3000 to train, 500 to test`, after a minute or two.

In [ ]:
def encode(messages):
    # A conversation (list of messages) -> its token ids, through the chat template, as in Lab 1.
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def response_length(messages):
    # Length of the last message, the response, in tokens. Used for the length baseline in Part 4.
    return len(tokenizer(messages[-1]["content"], add_special_tokens=False)["input_ids"])


def encode_pair(row):
    # One dataset row -> a dictionary with both conversations as token ids, the rating gap and both lengths.
    # Returns None for pairs we drop: too long, or two identical conversations.
    chosen, rejected = encode(row["chosen"]), encode(row["rejected"])
    if max(len(chosen), len(rejected)) > MAX_LENGTH or chosen == rejected:
        return None
    return {
        "chosen": chosen,
        "rejected": rejected,
        "gap": row["chosen-rating"] - row["rejected-rating"],
        "len_chosen": response_length(row["chosen"]),
        "len_rejected": response_length(row["rejected"]),
    }


t0 = time.time()
rows = dataset.shuffle(seed=SEED).select(range(3 * (N_TRAIN + N_TEST)))
pairs = [p for p in (encode_pair(row) for row in rows) if p is not None]
test_pairs = pairs[:N_TEST]
train_pairs = pairs[N_TEST:N_TEST + N_TRAIN]
print(f"Kept {len(pairs)} of {len(rows)} pairs ({time.time() - t0:.0f}s): {len(train_pairs)} to train, {len(test_pairs)} to test")
if len(train_pairs) < N_TRAIN:
    print("Fewer training pairs than N_TRAIN: increase MAX_LENGTH, or accept a smaller training set.")

A first statistic, before any model: **how often is the chosen response the longer one?**

**This cell:** counts the training pairs in which the chosen response has more tokens than the rejected one.

**You should see:** about 65%.

In [ ]:
longer = sum(p["len_chosen"] > p["len_rejected"] for p in train_pairs)
print(f"The chosen response is longer in {longer}/{len(train_pairs)} training pairs ({longer / len(train_pairs):.0%})")

If length were unrelated to the label, this would be close to 50%. It isn't. A model could reach that accuracy by
**counting words**, without understanding anything. This is the **verbosity bias** from the lecture, and Part 4 checks
whether our reward model learns it.

## Part 2: the reward model

The architecture is the one in the book's Chapter 5, [The default reward model architecture](https://rlhfbook.com/c/05-reward-models#the-default-reward-model-architecture):

1. Run the language model on the conversation, **without** its output layer. At every position it computes a hidden
   state, a vector of 1,024 numbers.
2. Take the hidden state at the **last real token** of the conversation. Because the model reads left to right, that
   position has seen the whole prompt and response.
3. A **linear head**, a single `nn.Linear(1024, 1)`, turns that vector into one number: the reward.

```text
conversation:  <|im_start|> user ... <|im_start|> assistant ... answer <|im_end|> \n  [pad] [pad]
hidden states:     h1        h2  ...      ...               ...                h_last
                                                                                 |
                                                                        linear head -> reward
```

### Exercise A: read the hidden state at the last real token

`last_token_hidden` must pick, for each conversation of a batch, **the hidden state at its last real token**. That one
vector of 1,024 numbers is what the head turns into the reward.

**Why the last token?** The model reads left to right, so the hidden state at a position only knows the tokens up to
that position. Only the **last** real token has read the whole prompt and the whole response, so it's the only one that
can summarize the full answer.

- **In:** `hidden`, shape $(B, L, H)$: $B$ conversations, padded to length $L$, with a vector of $H = 1{,}024$ numbers
  at every position. And `attention_mask`, shape $(B, L)$: 1 for real tokens, 0 for padding.
- **Out:** shape $(B, H)$, one vector per conversation.

**Why it's not simply the last position.** Conversations have different lengths, so the shorter ones are padded **on
the right**:

```text
position:          0   1   2   3   4    5     6
conversation 0:   tok tok tok tok tok [pad] [pad]     mask 1 1 1 1 1 0 0   ->  5 real tokens, last at 4
conversation 1:   tok tok tok tok tok  tok   tok      mask 1 1 1 1 1 1 1   ->  7 real tokens, last at 6
```

The last position, 6, is padding for conversation 0, and padding carries no meaning. So:

1. **Count** the real tokens of each row: the sum of its mask. The last real token is at **count − 1**.
2. **Pick** one position per row: `hidden[rows, positions]`, with one entry per row in `rows` and in `positions`.

<details><summary>Hint</summary>

If a row has $n$ real tokens, its last real token is at position $n - 1$. To pick one position per row, index with two
tensors: `hidden[rows, positions]`, where `rows = torch.arange(B)`.
</details>

<details><summary>Solution</summary>

```python
def last_token_hidden(hidden, attention_mask):
    last = attention_mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0), device=hidden.device), last]
```
</details>

**This cell:** is yours: complete the two `TODO` lines of `last_token_hidden`. Then run the next cell to check it.

In [ ]:
def last_token_hidden(hidden, attention_mask):
    # hidden: (B, L, H)   attention_mask: (B, L), 1 for real tokens and 0 for padding (on the right)
    # Return a (B, H) tensor: for each sequence, the hidden state at its last real token.

    # Step 1: the position of the last real token in each sequence.
    last = None  # TODO                            # (B,)

    # Step 2: pick hidden[b, last[b]] for every b.
    return None  # TODO                            # (B, H)

**This cell:** tests your function on a tiny, **made-up** batch, before we use it on the real model. The numbers are not from the model: they are just 0 to 23 in order, so you can tell by eye which position was picked. There are 2 sequences of 4 positions, each hidden state has 3 numbers instead of 1,024, and the mask, written by hand, says that sequence 0 has 3 real tokens and sequence 1 has 2.

**You should see:** `tensor([[ 6., 7., 8.], [15., 16., 17.]])`, the hidden states at positions 2 and 1, then `Exercise A: correct`. If you get `[9, 10, 11]` or `[21, 22, 23]`, your function picked padding.

In [ ]:
# Check your answer, on a tiny example you can work out by hand.
toy_hidden = torch.arange(24.0).reshape(2, 4, 3)        # 2 sequences, 4 positions, 3 numbers per position
toy_mask = torch.tensor([[1, 1, 1, 0],                  # 3 real tokens: the last one is at position 2
                         [1, 1, 0, 0]])                 # 2 real tokens: the last one is at position 1
picked = last_token_hidden(toy_hidden, toy_mask)
assert picked is not None, "Complete exercise A: some steps are still None."
print(picked)
assert torch.equal(picked, torch.stack([toy_hidden[0, 2], toy_hidden[1, 1]])), "Check the position of the last real token."
print("Exercise A: correct")

Now the reward model itself. It's short: a language model, your function, and one linear layer.

**This cell:** defines the reward model: the language model without its output layer (`backbone`), your `last_token_hidden`, and the linear head. Then it downloads Qwen3-0.6B (about 1.2 GB) and puts it on the GPU, in 16-bit, except the head, which stays in 32-bit because it's new and will be trained.

**You should see:** about 0.6B parameters, of which only 1,024 are in the head.

In [ ]:
class RewardModel(nn.Module):
    # A language model without its output layer, plus a linear head that turns one hidden state into one reward.
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone                                          # the language model, without its output layer
        self.head = nn.Linear(backbone.config.hidden_size, 1, bias=False)  # 1,024 numbers -> 1 reward

    def forward(self, input_ids, attention_mask):
        # (B, L) token ids and mask -> (B,) rewards, one per conversation.
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (B, L, H)
        return self.head(last_token_hidden(hidden, attention_mask)).squeeze(-1)                       # (B,)


# If this cell is run again, remove the old model and training state first, so that two copies don't fill the GPU.
for name in ["model", "optimizer", "scheduler", "scaler", "loss"]:
    globals().pop(name, None)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# On a T4, fp16 is many times faster than bf16 (see Lab 1). On a CPU we keep 32-bit floats.
dtype = torch.float16 if device == "cuda" else torch.float32
backbone = AutoModel.from_pretrained(MODEL_NAME, dtype=dtype)
model = RewardModel(backbone).to(device)
model.head.float()   # the new layer is trained, so it stays in 32 bits
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters; the head has {model.head.weight.numel()}")

### Batches of pairs

**This cell:** defines three helpers.

- `collate_pairs` stacks a batch of $B$ pairs as $2B$ conversations, **all the chosen first, then all the rejected**,
  padded on the right to the same length. For $B = 2$:

```text
row 0:  chosen of pair 0     tok tok tok tok [pad]
row 1:  chosen of pair 1     tok tok tok tok tok
row 2:  rejected of pair 0   tok tok tok [pad] [pad]
row 3:  rejected of pair 1   tok tok tok tok [pad]
```

- `forward_pairs` runs the model on that batch, which gives $2B$ rewards, and splits them back: the first $B$ are the
  chosen rewards, the last $B$ the rejected ones. So `r_chosen[k]` and `r_rejected[k]` belong to the same pair.
- `score_pairs` does this for a whole list of pairs, without training, and returns all the chosen and rejected rewards.
  To save time, it scores the pairs **from shortest to longest**, so that each batch holds conversations of similar
  length and little time is wasted on padding. The rewards come back in the original order.

**You should see:** no output.

In [ ]:
def collate_pairs(batch):
    # A list of B pairs -> token ids and attention mask of shape (2B, L): the B chosen conversations first, then the B rejected.
    sequences = [p["chosen"] for p in batch] + [p["rejected"] for p in batch]
    width = max(len(s) for s in sequences)                                          # pad everything to the longest
    input_ids = [s + [tokenizer.pad_token_id] * (width - len(s)) for s in sequences]   # padding on the right
    attention = [[1] * len(s) + [0] * (width - len(s)) for s in sequences]          # 1 = real token, 0 = padding
    return torch.tensor(input_ids, device=device), torch.tensor(attention, device=device)


def forward_pairs(batch):
    """Rewards of the chosen and the rejected responses of a batch of pairs."""
    input_ids, attention = collate_pairs(batch)
    with torch.amp.autocast(device, dtype=dtype, enabled=dtype != torch.float32):   # 16-bit on a GPU, as in Lab 1
        rewards = model(input_ids, attention).float()                               # (2B,) rewards
    return rewards[:len(batch)], rewards[len(batch):]                               # split back: chosen, rejected


def pair_length(p):
    # The longer of the two conversations of a pair, in tokens.
    return max(len(p["chosen"]), len(p["rejected"]))


@torch.no_grad()                                   # no gradients: we only evaluate
def score_pairs(pair_list, batch_size=8):
    # Rewards for a list of pairs, in batches. Returns two (N,) tensors on the CPU: chosen rewards, rejected rewards,
    # in the same order as pair_list.
    model.eval()
    # Speed: score the pairs from shortest to longest, so that each batch holds conversations of similar length and
    # little time is wasted on padding. The rewards are put back in the original order at the end.
    order = sorted(range(len(pair_list)), key=lambda k: pair_length(pair_list[k]))
    chosen, rejected = torch.zeros(len(pair_list)), torch.zeros(len(pair_list))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        r_c, r_r = forward_pairs([pair_list[k] for k in idx])
        chosen[idx], rejected[idx] = r_c.cpu(), r_r.cpu()
        done = min(start + batch_size, len(order))
        print(f"\r  scored {done}/{len(order)} pairs", end="")   # progress, on one line
    print()
    return chosen, rejected

### The Bradley-Terry loss, on a tiny example

For one pair, the loss is $-\log \sigma(\text{margin})$, with margin $= r_{\text{chosen}} - r_{\text{rejected}}$. Before
running the cell, **predict** which of these three pairs costs the most, and which costs almost nothing:

| Pair | $r_{\text{chosen}}$ | $r_{\text{rejected}}$ | Margin |
|---|---|---|---|
| 1 | 3 | 0 | 3: the model agrees with the label, confidently |
| 2 | 1 | 1 | 0: the model can't tell |
| 3 | 0 | 3 | $-3$: the model disagrees, confidently |

**This cell:** computes $\sigma(\text{margin})$ and the loss for the three pairs of the table, in plain Python.

**You should see:** 0.049, 0.693 and 3.049. Did you predict right?

In [ ]:
for r_c, r_r in [(3.0, 0.0), (1.0, 1.0), (0.0, 3.0)]:
    m = r_c - r_r
    p = 1 / (1 + math.exp(-m))              # sigma(margin): the probability the model gives to the label
    print(f"margin {m:+.0f}:  sigma = {p:.3f}   loss = -log(sigma) = {-math.log(p):.3f}")

A confident **correct** ranking costs almost nothing (0.049). A coin flip costs $\log 2 \approx 0.693$. A confident
**wrong** ranking costs a lot (3.049), and the gradient pushes hard to fix it. Training only ever sees which response
won, never a "true" reward.

### Exercise B: the Bradley-Terry loss

Training maximizes the likelihood of the observed choices, as in the lecture. For a batch of pairs, the loss is the
average negative log-likelihood:

$$
\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \log \sigma\left(r_{\text{chosen}, i} - r_{\text{rejected}, i}\right).
$$

Use `F.logsigmoid` rather than `torch.log(torch.sigmoid(...))`: for a very negative margin, `sigmoid` rounds to 0 and
the log becomes $-\infty$, while `logsigmoid` computes the same value safely.

<details><summary>Hint</summary>

One line: the margin, then `F.logsigmoid`, then the mean, with a minus sign.
</details>

<details><summary>Solution</summary>

```python
def bt_loss(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected).mean()
```
</details>

**This cell:** is yours: write `bt_loss`, in one line. Then run the next cell to check it.

In [ ]:
def bt_loss(r_chosen, r_rejected):
    # r_chosen, r_rejected: (B,) rewards of the chosen and rejected responses.
    # TODO: return the Bradley-Terry loss, averaged over the batch (a single number).
    raise NotImplementedError("Complete exercise B")

**This cell:** checks three things: equal rewards give $\log 2 \approx 0.693$, a coin flip; two pairs give the right average; and a huge wrong margin gives a finite loss. That last one is what `F.logsigmoid` guarantees and `torch.log(torch.sigmoid(...))` doesn't.

**You should see:** `Exercise B: correct`, then the losses 0.049, 0.693 and 3.049.

In [ ]:
# Check your answer
assert abs(bt_loss(torch.tensor([0.0]), torch.tensor([0.0])).item() - math.log(2)) < 1e-6, "Equal rewards: a coin flip, loss log 2."
assert abs(bt_loss(torch.tensor([2.0, 1.0]), torch.tensor([0.0, 1.0])).item() - (0.1269 + 0.6931) / 2) < 1e-3
assert torch.isfinite(bt_loss(torch.tensor([-200.0]), torch.tensor([200.0]))), "Use F.logsigmoid, which is stable."
print("Exercise B: correct")
print(f"Loss for margins 3, 0 and -3: {bt_loss(torch.tensor([3.0]), torch.tensor([0.0])):.3f}, "
      f"{bt_loss(torch.tensor([0.0]), torch.tensor([0.0])):.3f}, {bt_loss(torch.tensor([0.0]), torch.tensor([3.0])):.3f}")

A correct ranking with a large margin costs almost nothing; a wrong one with a large margin costs a lot.

### Measure first

The head starts with random weights, so the untrained model should be right roughly half the time. It can do a little
better or worse: even a random head reacts to surface features of the text, such as its length. To say how sure we are
of an accuracy measured on 500 pairs, we use the **Wilson interval** you wrote in Lab 1.

**This cell:** defines the Wilson interval (from Lab 1) and `report_accuracy`, then scores the 500 test pairs with the **untrained** model and prints its accuracy. It takes about a minute.

**You should see:** a progress line, `scored 500/500 pairs`, then an accuracy somewhere between 0.5 and 0.6, with an interval about $\pm 0.04$ wide. The text after the output explains why it is not exactly 0.5.

In [ ]:
def wilson_interval(successes, n, z=1.96):
    # 95% Wilson interval for a proportion: k successes out of n (you wrote it in Lab 1).
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half


def report_accuracy(name, correct):
    # correct: one True/False per pair. Prints the accuracy with its 95% Wilson interval.
    k, n = int(sum(correct)), len(correct)
    low, high = wilson_interval(k, n)
    print(f"{name}: accuracy {k / n:.3f}  ({k}/{n}), 95% interval [{low:.3f}, {high:.3f}]")


t0 = time.time()
before_c, before_r = score_pairs(test_pairs)
report_accuracy("Before training", (before_c > before_r).tolist())
print(f"({time.time() - t0:.0f}s to score {len(test_pairs)} pairs)")

### Making sense of this number

**Why isn't it exactly 0.5?** The head is random, but it multiplies hidden states that already carry information about
the text, its length for example. So a random head can lean towards longer responses by chance, and on this data, longer
usually wins. Look at the interval: it tells you whether the difference from 0.5 could be chance alone.

Write the number down. Part 4 compares it with the trained model, on the **same** 500 pairs.

## Part 3: train

Start the training cell and let it run while you discuss. It stops by itself after `MAX_MINUTES`.

**How we fit this on a free GPU.** We train only the last 8 of the 28 layers and the new head; the rest stay frozen. As
in Lab 1, the frozen weights use 16-bit floats, the trained weights use 32 bits, and a gradient scaler keeps small
16-bit gradients from rounding to zero. The book's run ([`code/reward_models`](https://github.com/natolambert/rlhf-book/tree/main/code/reward_models)) trains the whole model, for 2 epochs on 5,000 pairs.

### What happens in one training step

1. **Forward pass:** the model reads a batch of 4 pairs, so 8 conversations, and outputs 8 rewards: 4 for the chosen
   responses and 4 for the rejected ones.
2. **Loss:** your `bt_loss` on the 4 margins. This is the only change from Lab 1, where the loss was next-token
   cross-entropy.
3. **Backward pass:** PyTorch computes the gradient of the loss for every trainable parameter.
4. **Accumulate:** steps 1 to 3 repeat 4 times, adding up the gradients, so that each update uses 16 pairs.
5. **Update:** AdamW moves the parameters a little against the gradient, after **clipping** it to a maximum size.

The learning rate rises over the first 10% of steps (**warm-up**), then decreases linearly to 0, as in Lab 1.

**While it trains,** discuss:

1. The model never sees a "correct" reward, only which of two responses won. What does that imply about the scale of the
   rewards? (Think about identification, in the lecture.)
2. What could the model learn that would raise its training accuracy without making it a better judge of quality?

<details><summary>Possible answers</summary>

1. **Only differences are identified.** Adding the same constant to every reward changes no margin, so no loss. The
   rewards' zero is arbitrary, and a reward of $+2$ means nothing on its own, only compared with another response to the
   same prompt.
2. **Shortcuts that correlate with the label:** length, lists and formatting, a confident tone, or the style of the
   model that wrote the response. In Part 1 you saw that the chosen response is usually longer, so counting tokens already
   wins most of the time.
</details>

**This cell:** trains the reward model.

1. **Freeze** every layer except the last 8 and the head, and print how many parameters are trained.
2. **Set up** the optimizer, the gradient scaler and the learning-rate schedule.
3. **Loop** over the training pairs once, printing every 10 optimizer steps. Each step is the one described above.
   To save time, each batch holds pairs of **similar length** (less padding), and the batches come in a random order.
4. **Stop** at `MAX_MINUTES`, free the GPU memory, and plot the loss, the training accuracy and the mean margin.

**You should see:** `Training ...M of ...M parameters`, then lines like `step 10/187  loss 0.68  accuracy 0.55  margin +0.05`. Over time, the loss should fall below the dashed line at 0.693, the accuracy should rise above 0.5, and the margin should grow above 0.

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
model.train()

# Freeze everything except the head, the final normalization and the last layers.
for param in model.parameters():
    param.requires_grad = False
for module in [model.head, model.backbone.norm, *model.backbone.layers[-TRAIN_LAST_LAYERS:]]:
    for param in module.parameters():
        param.requires_grad = True
        if dtype == torch.float16:
            param.data = param.data.float()
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Training {sum(p.numel() for p in trainable) / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

def make_batches(pairs):
    # One epoch of training batches. Speed: pairs are shuffled, then grouped into chunks of 50 batches; inside a chunk,
    # pairs of similar length go into the same batch, so there is little padding. The batches are shuffled at the end,
    # so training still sees them in a random order.
    order = random.sample(range(len(pairs)), len(pairs))
    chunk = 50 * BATCH_SIZE
    batches = []
    for start in range(0, len(order), chunk):
        part = sorted(order[start:start + chunk], key=lambda k: pair_length(pairs[k]))
        batches += [part[b:b + BATCH_SIZE] for b in range(0, len(part) - BATCH_SIZE + 1, BATCH_SIZE)]
    random.shuffle(batches)
    return batches


micro_batches_per_epoch = len(train_pairs) // BATCH_SIZE
total_steps = EPOCHS * micro_batches_per_epoch // ACCUM_STEPS
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0, foreach=False)
scaler = torch.amp.GradScaler(device, enabled=dtype == torch.float16)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

history = {"loss": [], "accuracy": [], "margin": []}
running = {"loss": 0.0, "correct": 0, "margin": 0.0}
step, start, finished = 0, time.time(), False
optimizer.zero_grad()

for epoch in range(EPOCHS):
    batches = make_batches(train_pairs)
    for i, batch_idx in enumerate(batches):
        batch = [train_pairs[j] for j in batch_idx]
        r_chosen, r_rejected = forward_pairs(batch)
        loss = bt_loss(r_chosen, r_rejected)
        scaler.scale(loss / ACCUM_STEPS).backward()
        running["loss"] += loss.item() / ACCUM_STEPS
        running["correct"] += (r_chosen > r_rejected).sum().item()
        running["margin"] += (r_chosen - r_rejected).sum().item()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            n_pairs = BATCH_SIZE * ACCUM_STEPS
            history["loss"].append(running["loss"])
            history["accuracy"].append(running["correct"] / n_pairs)
            history["margin"].append(running["margin"] / n_pairs)
            running = {"loss": 0.0, "correct": 0, "margin": 0.0}
            if step % 10 == 0 or step == 1:
                recent = history["accuracy"][-10:]
                print(f"step {step:4d}/{total_steps}  loss {history['loss'][-1]:.3f}  "
                      f"accuracy (last 10 steps) {sum(recent) / len(recent):.2f}  "
                      f"margin {history['margin'][-1]:+.2f}  elapsed {(time.time() - start) / 60:5.1f} min")
            if (time.time() - start) / 60 > MAX_MINUTES:
                print("Time limit reached: stopping early, which is fine for this lab.")
                finished = True
                break
    if finished:
        break

minutes = (time.time() - start) / 60
if device == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
del optimizer, scheduler, scaler, loss, r_chosen, r_rejected
model.zero_grad(set_to_none=True)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print(f"Done: {step} optimizer steps in {minutes:.1f} min")

window = 20
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, key, label in zip(axes, ["loss", "accuracy", "margin"], ["Loss", "Training accuracy", "Mean margin"]):
    values = history[key]
    smoothed = [sum(values[max(0, i - window + 1):i + 1]) / len(values[max(0, i - window + 1):i + 1]) for i in range(len(values))]
    ax.plot(values, alpha=0.3)
    ax.plot(smoothed)
    ax.set_xlabel("Optimizer step")
    ax.set_title(label)
axes[0].axhline(math.log(2), color="gray", linestyle="--", linewidth=1)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

The dashed lines are the coin flip: a loss of $\log 2 \approx 0.693$ and an accuracy of 0.5. Each step sees only 16
pairs, so the curves are noisy. The held-out accuracy below is the number that counts.

## Part 4: what did the model learn?

### Held-out accuracy, against a simple baseline

A reward model is only useful if it does better than a rule anyone could write. The obvious rule here is **"the longer
response wins"**. Both are measured on the same 500 test pairs.

**This cell:** scores the 500 test pairs with the **trained** model, and applies the length rule to the same pairs. It prints three accuracies with their intervals: before training, after training, and "the longer response wins".

**You should see:** the trained accuracy clearly above the untrained one. Whether it beats the length rule is the question of exercise C.

In [ ]:
after_c, after_r = score_pairs(test_pairs)
margin = after_c - after_r
rm_correct = (margin > 0).tolist()
length_correct = [p["len_chosen"] > p["len_rejected"] for p in test_pairs]

report_accuracy("Before training     ", (before_c > before_r).tolist())
report_accuracy("After training      ", rm_correct)
report_accuracy("Longer response wins", length_correct)

### Exercise C: compare them fairly, pair by pair

Both methods were scored on the **same** 500 pairs, so their accuracies are not independent: on many pairs both are
right, and on some both are wrong. Those pairs say **nothing** about which method is better. Only the pairs where they
**disagree** do, as in Lab 1's paired comparison.

| | Length rule right | Length rule wrong |
|---|---|---|
| **Reward model right** | both right: no information | **only the reward model** |
| **Reward model wrong** | **only the length rule** | both wrong: no information |

Complete `paired_counts`: it returns the two counts in bold.

<details><summary>Hint</summary>

Walk through the two lists together with `zip`. A pair counts for A when `a and not b`. `sum` of booleans counts the
`True` values.
</details>

<details><summary>Solution</summary>

```python
def paired_counts(right_a, right_b):
    only_a = sum(a and not b for a, b in zip(right_a, right_b))
    only_b = sum(b and not a for a, b in zip(right_a, right_b))
    return only_a, only_b
```
</details>

**This cell:** is yours: complete the two `None` lines of `paired_counts`. Then run the next cell to check it.

In [ ]:
def paired_counts(right_a, right_b):
    # right_a, right_b: two lists of True/False, one entry per test pair, for two methods A and B.
    # Return (only_a, only_b): the number of pairs that A gets right and B gets wrong, and the reverse.
    only_a = None  # TODO
    only_b = None  # TODO
    return only_a, only_b

**This cell:** checks your function on 5 made-up pairs, then applies it to the reward model and the length rule.

**You should see:** `Exercise C: correct`, then the two counts. Read them as explained below.

In [ ]:
# Check your answer, on a tiny example you can work out by hand.
toy = paired_counts([True, True, False, False, True], [True, False, True, False, False])
assert None not in toy, "Complete exercise C: some counts are still None."
assert tuple(toy) == (2, 1), f"Expected (2, 1): A alone is right on pairs 2 and 5, B alone on pair 3. Got {toy}."
print("Exercise C: correct")

only_rm, only_length = paired_counts(rm_correct, length_correct)
print(f"Right only for the reward model: {only_rm}. Right only for the length rule: {only_length}.")

**How to read this.** If the two intervals overlap, that alone doesn't settle the question; the paired counts are more
informative. If the reward model were no better than the length rule, the disagreements would split about 50/50 between
the two. A split like 60 against 25 is strong evidence; 45 against 40 is not. The book's run trains the whole model on
more data, so expect it to do better than ours.

### Are larger rating gaps easier?

Under Bradley-Terry, a larger difference in utility means a more predictable choice. The rating gap, chosen rating minus
rejected rating, is a rough measure of that difference. If the reward model learned something real, it should be more
accurate when the gap is large.

**This cell:** splits the test pairs into three groups by rating gap, computes the accuracy and its interval in each group, and draws a bar chart.

**You should see:** three bars. If the model learned something real, they rise from left to right. The groups are small, so look at the intervals before concluding.

In [ ]:
bins = [(0, 0.75, "small gap (at most 0.75)"), (0.75, 2.0, "medium gap (0.75 to 2]"), (2.0, 5.0, "large gap (above 2)")]
names, rates, errors = [], [], []
for low_gap, high_gap, name in bins:
    chosen_idx = [i for i, p in enumerate(test_pairs) if low_gap < p["gap"] <= high_gap]
    k = sum(rm_correct[i] for i in chosen_idx)
    n = len(chosen_idx)
    low, high = wilson_interval(k, n)
    print(f"{name:<26} n = {n:3d}   accuracy {k / n:.2f}   95% interval [{low:.2f}, {high:.2f}]")
    names.append(name.split(" (")[0])
    rates.append(k / n)
    errors.append([k / n - low, high - k / n])

plt.bar(names, rates, yerr=list(zip(*errors)), capsize=6, color="#1B2A49")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.ylabel("Held-out accuracy")
plt.ylim(0, 1)
plt.title("Reward model accuracy by rating gap")
plt.show()

### Is the model calibrated?

The model's margin also gives a probability: Bradley-Terry says the chosen response wins with probability
$\sigma(\text{margin})$. Its **confidence** in its own pick is $\sigma(|\text{margin}|)$, between 0.5 and 1. A
**calibrated** model is right 70% of the time on the pairs where its confidence is about 0.7.

We group the test pairs by confidence and compare, in each group, the average confidence with the accuracy. A calibrated
model lies on the diagonal.

**This cell:** computes the model's confidence on each test pair, groups the pairs by confidence (0.5 to 0.6, 0.6 to 0.7, and so on), and compares the mean confidence with the accuracy in each group, as a table and as a plot.

**You should see:** points close to the dashed diagonal if the model is calibrated.

In [ ]:
confidence = torch.sigmoid(margin.abs())
edges = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0001]
mean_conf, mean_acc, sizes = [], [], []
print(f"{'confidence':<12}{'pairs':>6}{'mean confidence':>18}{'accuracy':>10}")
for lo, hi in zip(edges[:-1], edges[1:]):
    in_bin = [i for i in range(len(test_pairs)) if lo <= confidence[i] < hi]
    if not in_bin:
        continue
    c = sum(confidence[i].item() for i in in_bin) / len(in_bin)
    a = sum(rm_correct[i] for i in in_bin) / len(in_bin)
    print(f"[{lo:.1f}, {min(hi, 1):.1f})  {len(in_bin):6d}{c:18.2f}{a:10.2f}")
    mean_conf.append(c)
    mean_acc.append(a)
    sizes.append(len(in_bin))

plt.plot([0.5, 1], [0.5, 1], color="gray", linestyle="--", label="Perfect calibration")
plt.scatter(mean_conf, mean_acc, s=[3 * n for n in sizes], color="#D98E04", label="Reward model (area = number of pairs)")
plt.xlabel("Mean confidence")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Calibration on held-out pairs")
plt.show()

Points **below** the diagonal mean the model is **overconfident**: it's right less often than it claims. Points
**above** mean it is **underconfident**. Bins with few pairs are noisy, so don't read too much into them.

Why calibration matters: if the reward model says 0.9, the chosen response should win about 90% of the time. When
the reward is used to train a policy, its size, not just its sign, drives the updates.

### Did the model learn the length bias?

We saw in Part 1 that the chosen response is usually the longer one. The sharpest test splits the test pairs in two:
pairs where the chosen response is **longer**, and pairs where it is **shorter**. A model that only counts words is
right on the first group and wrong on the second.

**This cell:** splits the test pairs in two: those where the chosen response is longer, and those where it's shorter, and prints the accuracy on each. Then it compares how often the labels, and the model, prefer the longer response.

**You should see:** two accuracies. A large gap between them means the model relies on length.

In [ ]:
chosen_longer = [i for i, p in enumerate(test_pairs) if p["len_chosen"] > p["len_rejected"]]
chosen_shorter = [i for i, p in enumerate(test_pairs) if p["len_chosen"] < p["len_rejected"]]
report_accuracy("Chosen is longer ", [rm_correct[i] for i in chosen_longer])
report_accuracy("Chosen is shorter", [rm_correct[i] for i in chosen_shorter])

# How often does each one prefer the longer response?
pairs_with_lengths = chosen_longer + chosen_shorter
labels_prefer_longer = len(chosen_longer) / len(pairs_with_lengths)
longer_set = set(chosen_longer)
rm_prefers_longer = sum(bool(margin[i] > 0) == (i in longer_set) for i in pairs_with_lengths) / len(pairs_with_lengths)
print(f"\nThe labels prefer the longer response in {labels_prefer_longer:.0%} of pairs; the reward model in {rm_prefers_longer:.0%}.")

**How to read this.**

- If accuracy is much lower when the chosen response is shorter, the model relies on length.
- If the model prefers the longer response **more often than the labels do**, it has amplified the bias, not just
  copied it. This is the **systematic noise** of the lecture: it's in the data, more data won't remove it, and a policy
  trained against this reward in step 3 would learn to write longer answers.

## Part 5: score your own responses

`score` gives the reward of one response to one prompt. Rewards only mean something **relative to each other**, for the
same prompt: only differences are identified, as in the lecture.

**This cell:** defines `score`, the reward of one response to one prompt, then scores three responses to each of two prompts.

**You should see:** three rewards per prompt. Compare them **within** a prompt, never across prompts.

In [ ]:
@torch.no_grad()
def score(prompt, response):
    # The reward of one response to one prompt. Only differences between responses to the same prompt mean anything.
    model.eval()
    messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]   # a one-turn conversation
    ids = torch.tensor([encode(messages)], device=device)                                          # a batch of one, no padding
    with torch.amp.autocast(device, dtype=dtype, enabled=dtype != torch.float32):
        return model(ids, torch.ones_like(ids)).item()


tests = [
    ("What is the capital of Australia?", [
        "The capital of Australia is Canberra.",
        "The capital of Australia is Sydney.",
        "Great question! Australia is a fascinating country with a rich history and diverse landscapes. Its capital, "
        "Sydney, is famous for its Opera House and Harbour Bridge, and it attracts millions of visitors every year.",
    ]),
    ("Give me one tip to sleep better.", [
        "Go to bed and wake up at the same time every day, even on weekends.",
        "I'm sorry, but I can't help with that.",
        "Sleep.",
    ]),
]
for prompt, responses in tests:
    print("PROMPT:", prompt)
    for response in responses:
        print(f"  {score(prompt, response):+7.2f}  {response[:90]}")
    print()

**Your turn.** Edit the prompts and responses above, then discuss:

1. Does the model prefer the correct answer to the wrong one? Does it prefer a **long, fluent, wrong** answer to a short,
   correct one?
2. Try to **fool** it: find a bad response that scores higher than a good one. What trick worked? This is reward
   hacking, on a small scale: an RL policy trained against this model would find such tricks by itself.

<details><summary>Possible answers</summary>

Tricks that often work on small reward models:

- **Length and detail:** a long, fluent answer with a wrong fact can beat a short correct one.
- **Confident or enthusiastic tone:** "Great question!", "Certainly!".
- **Formatting:** bullet points and headings.
- **Restating the question** at length before answering.

What usually does **not** fool it is an outright refusal, or a one-word answer. If your model prefers "Sydney" to
"Canberra", it has learned style more than facts, which is expected from 15 minutes of training on 3,000 pairs.
</details>

## What you learned

- **A reward model** is a language model with a linear head, read at the last real token of the conversation
- **Training it is Bradley-Terry maximum likelihood,** with a neural network as the utility: the loss is
  $-\log \sigma(r_{\text{chosen}} - r_{\text{rejected}})$
- **Evaluate it like any statistical model:** held-out accuracy with an interval, a simple baseline, paired comparisons and
  calibration
- **It learns what's in the labels,** including their biases: here, a preference for longer responses
- **Rewards are relative:** only differences between responses to the same prompt mean anything

**Discuss before leaving:**

1. Is held-out accuracy a good measure of a reward model? What else would you want to know before training a policy
   against it?
2. The labels come from GPT-4. Whose preferences has our model learned?
3. We dropped the longest 40% of pairs. How could that change what the model learned, and the accuracy we measured?

<details><summary>Possible answers</summary>

1. **Not on its own.** A reward model will be pushed by RL towards responses it has never seen, so we'd also want its
   accuracy on **new kinds of responses**, its calibration, and checks for exploitable biases like length.
2. **GPT-4's preferences,** as prompted by UltraFeedback's rating instructions, not those of any group of people.
3. The model never saw **long responses**, so its scores for them are an extrapolation, and our accuracy only describes
   short pairs. The length bias could also look different on long pairs.
</details>

## Going further (optional)

These are for after class, if you want to go deeper. Change one thing at a time, and run the notebook again from the top.

- **HW1, the Bradley-Terry model from scratch:** maximum likelihood, Fisher information and the coverage of confidence
  intervals, where the truth is known. Not graded.
- **Frozen layers:** vary `TRAIN_LAST_LAYERS` (0, 2, 8). With 0, only the head is trained, a logistic regression on
  frozen features. How much do the extra layers add?
- **Dataset size:** vary `N_TRAIN` (500, 1,000, 3,000), and plot held-out accuracy with its interval.
- **Margin loss:** use the rating gap in the loss, $-\log \sigma(r_c - r_r - m \cdot \text{gap})$, as in Llama 2
  ([Lambert, Chapter 5](https://rlhfbook.com/c/05-reward-models#preference-margin-loss)). Does accuracy on small-gap pairs change?
- **Length control:** can you remove the length bias, for example by training only on pairs where the two responses have
  similar lengths? What does it cost in accuracy?
- **Reading:** Chapter 5 of the book, and its [full code](https://github.com/natolambert/rlhf-book/tree/main/code/reward_models).